## Setup

In [1]:
import os

# Set before importing datasets/transformers; restart the kernel if already imported.
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "30"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "10"

import random
import torch
import pandas as pd
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
def set_seed(seed=42): # 42 based on the "Answer to the Ultimate Question of Life, the Universe, and Everything" from The Hitchhiker's Guide to the Galaxy
    random.seed(seed)
    torch.manual_seed(seed)

set_seed()

In [2]:
mistral7b = 'mistralai/Mistral-7B-v0.1'
# STEP 1. Check and make sure you're using the right model and notebook here.
model_name = mistral7b

## EDA

In [3]:
df = pd.read_csv("frankenstein_chunks.csv")
df.head()

,text
0,﻿The Project Gutenberg eBook of Frankenstein; ...
1,Further corrections by Menno de Leeuw.\n\n\n**...
2,"I am already far north of London, and as I wal..."
3,Its productions and features may be without ex...
4,But supposing all these conjectures to be fals...


In [4]:
print("Dataframe Info:")
print(df.info())
print("\n")
print("Dataframe Description:")
print(df.describe())
print("\n")
print("Number of unique values in each column:")
print(df.nunique())
random_index= random.randint(0, len(df) - 1)
df.loc[random_index, 'text']

Dataframe Info:
<class 'pandas.DataFrame'>
RangeIndex: 481 entries, 0 to 480
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   text    481 non-null    str  
dtypes: str(1)
memory usage: 434.3 KB
None


Dataframe Description:
                                                     text
count                                                 481
unique                                                481
top     ﻿The Project Gutenberg eBook of Frankenstein; ...
freq                                                    1


Number of unique values in each column:
text    481
dtype: int64


'The thatch had fallen in, the walls were unplastered, and the\ndoor was off its hinges. I ordered it to be repaired, bought some\nfurniture, and took possession, an incident which would doubtless have\noccasioned some surprise had not all the senses of the cottagers been\nbenumbed by want and squalid poverty. As it was, I lived ungazed at\nand unmolested, hardly thanked for the pittance of food and clothes\nwhich I gave, so much does suffering blunt even the coarsest sensations\nof men.\n\nIn this retreat I devoted the morning to labour; but in the evening,\nwhen the weather permitted, I walked on the stony beach of the sea to\nlisten to the waves as they roared and dashed at my feet. It was a\nmonotonous yet ever-changing scene. I thought of Switzerland; it was\nfar different from this desolate and appalling landscape. '

In [5]:
df.isnull().sum()

text    0
dtype: int64

In [6]:
# Keep the test set out of checkpoint selection and hyperparameter tuning.
from sklearn.model_selection import train_test_split
train_df, heldout_df = train_test_split(df, test_size=0.2, random_state=42)
validation_df, test_df = train_test_split(heldout_df, test_size=0.5, random_state=42)

# STEP 2. Convert the fixed 384/48/49 splits to Hugging Face Datasets.
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
validation_dataset = Dataset.from_pandas(validation_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_df, preserve_index=False)


## Model Import and Tokenization

Restart the kernel and run from Setup so the HTTP download settings take effect before Hugging Face is imported. HTTP is used here to bypass the Xet transfer seen in the stalled run; this is a transport workaround, not a proven diagnosis.

Mistral-7B downloads about **14.48 GB (13.49 GiB)** of full-precision safetensors before quantizing on the GPU. Four-bit loading reduces VRAM usage, not this initial download. Allow sufficient cache disk space; the first run can take many minutes depending on bandwidth. Later runs reuse cached files, and interrupted HTTP downloads can resume when rerun.

Cell 10 separates download from local-only GPU loading, with timings for each stage. It downloads the model configuration, safetensors and tokenizer, with two concurrent file transfers. The tokenizer then loads from this same snapshot without another Hub request. The 30-second setting is a network read timeout (with library retries), not a limit on total cell runtime. CUDA is required; all weights stay on GPU rather than silently offloading to CPU. PyTorch may report that optional Triton FLOP counting is unavailable on this Windows setup; this does not prevent SDPA training, and the notice is not suppressed.

In [7]:
from time import perf_counter # Import the perf_counter function for measuring elapsed time
from huggingface_hub import snapshot_download # Import the snapshot_download function from the Hugging Face Hub

if not torch.cuda.is_available(): # Check if CUDA is available
    raise RuntimeError("CUDA is required for this QLoRA notebook. Select a CUDA-enabled kernel before downloading.")

free_vram, total_vram = torch.cuda.mem_get_info(0) # Get the free and total VRAM on the GPU
print(f"GPU: {torch.cuda.get_device_name(0)} | Free VRAM: {free_vram / 2**30:.2f}/{total_vram / 2**30:.2f} GiB", flush=True) # Print the GPU name and its free/total VRAM

quant_config = BitsAndBytesConfig( # Configure the model to use 4-bit NF4 quantization on the GPU
    # STEP 3. Quantize the model to 4-bit NF4 on the GPU.
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

started = perf_counter() # Record the start time for the download/cache check
print(f"Stage 1/2: downloading {model_name} (or reusing the Hub cache)...", flush=True) # Inform the user that the download/cache check is starting
model_path = snapshot_download( # Download the model from the Hugging Face Hub or reuse the cache
    repo_id=model_name,
    allow_patterns=["config.json", "generation_config.json", "*.safetensors", "*.safetensors.index.json", "tokenizer*", "special_tokens_map.json"],
    max_workers=2, # Set the maximum number of worker threads for downloading the model
)
print(f"Download/cache check finished in {perf_counter() - started:.1f}s. Cache: {model_path}", flush=True) # Print the time taken for the download/cache check and the cache path

started = perf_counter() # Record the start time for loading and quantizing the model
print("Stage 2/2: loading cached weights and quantizing on cuda:0...", flush=True) # Inform the user that the model loading and quantization is starting
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=quant_config,
    device_map={"": 0}, # Map the model to the first CUDA device (cuda:0)
    dtype=torch.float16,
    attn_implementation="sdpa",
    use_cache=False,
    use_safetensors=True,
    local_files_only=True,
)
print(f"GPU loading finished in {perf_counter() - started:.1f}s.", flush=True) # Print the time taken for loading and quantizing the model
print(f"Model is running on: {model.device}") # Print the device the model is running on
print(f"Model memory footprint: {model.get_memory_footprint() / 2**30:.2f} GiB") # Print the model's memory footprint on the GPU

GPU: NVIDIA GeForce RTX 2070 Super | Free VRAM: 6.96/8.00 GiB
Stage 1/2: downloading mistralai/Mistral-7B-v0.1 (or reusing the Hub cache)...


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Download/cache check finished in 0.2s. Cache: C:\Users\nomadant\.cache\huggingface\hub\models--mistralai--Mistral-7B-v0.1\snapshots\27d67f1b5f57dc0953326b2601d68371d40ea8da
Stage 2/2: loading cached weights and quantizing on cuda:0...


W1020 22:51:54.961000 30676 site-packages\torch\utils\flop_counter.py:113] triton not found; flop counting will not work for triton kernels


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

GPU loading finished in 34.6s.
Model is running on: cuda:0
Model memory footprint: 3.74 GiB


In [8]:
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, TaskType

# STEP 4. Prepare the model for QLoRA. Configure LoRA for our finetuning run. Then tokenize the data.
model = prepare_model_for_kbit_training(
    model, gradient_checkpointing_kwargs={"use_reentrant": False}
)
config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM, # Specify the task type for LoRA
)

model = get_peft_model(model, config)
model.print_trainable_parameters()
tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
max_length = 384

def tokenize_function(examples):
    encoded = tokenizer(examples["text"], truncation=True, max_length=max_length - 1)
    encoded["input_ids"] = [ids + [tokenizer.eos_token_id] for ids in encoded["input_ids"]]
    encoded["attention_mask"] = [mask + [1] for mask in encoded["attention_mask"]]
    return encoded

tokenized_train_dataset = train_dataset.map(tokenize_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_validation_dataset = validation_dataset.map(tokenize_function, batched=True, remove_columns=validation_dataset.column_names)
tokenized_test_dataset = test_dataset.map(tokenize_function, batched=True, remove_columns=test_dataset.column_names)

def causal_lm_collator(features):
    batch = tokenizer.pad(features, padding=True, pad_to_multiple_of=8, return_tensors="pt")
    labels = batch["input_ids"].clone()
    # PAD and EOS share an ID: mask by attention, not token ID, to retain real EOS labels.
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels
    return batch

Could not reach the Hub ([WinError 10054] An existing connection was forcibly closed by the remote host). Using cached commit hash for 'mistralai/Mistral-7B-v0.1'.


trainable params: 13,631,488 || all params: 7,255,363,584 || trainable%: 0.1879


Map:   0%|          | 0/384 [00:00<?, ? examples/s]

Map:   0%|          | 0/48 [00:00<?, ? examples/s]

Map:   0%|          | 0/49 [00:00<?, ? examples/s]

## Base Model Evaluation

In [9]:
# Use the same seeded decoding policy for the base and finetuned model.

def generate_text(prompt: str) -> str:
    set_seed()
    model.eval()
    model.gradient_checkpointing_disable()
    model.config.use_cache = True
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs, max_new_tokens=100, do_sample=True,
            temperature=0.8, top_p=0.9, top_k=50,
            repetition_penalty=1.1, no_repeat_ngram_size=4,
            pad_token_id=tokenizer.pad_token_id, eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [10]:
# STEP 5. Generate a completion with the base model for informal evaluation.
base_generation = generate_text("I'm afraid I've created a ")
base_generation

"I'm afraid I've created a 4-year old monster.\n\nFor the past 2 years, my son has played with his brother's toys - mostly Duplo and Lego. He loves to build and create things. He's never really had his own set of Lego or Duplo but recently he started asking for Lego. We gave him some small sets at Christmas (some that were on sale) but he loved them so much he wanted more. So, we got him a big box of Leg"

In [11]:
import math

def calc_perplexity(eval_model, dataset: Dataset) -> float:
    eval_model.eval()
    eval_model.gradient_checkpointing_disable()
    total_nll = 0.0
    total_tokens = 0
    with torch.inference_mode():
        for row in dataset:
            batch = causal_lm_collator([row])
            batch = {name: value.to(eval_model.device) for name, value in batch.items()}
            outputs = eval_model(**batch, use_cache=False)
            # Causal LM loss predicts labels[1:], not the first token or padding.
            count = batch["labels"][:, 1:].ne(-100).sum().item()
            total_nll += outputs.loss.item() * count
            total_tokens += count
    if total_tokens == 0:
        raise ValueError("Perplexity requires at least one unmasked target token.")
    mean_nll = total_nll / total_tokens
    if not math.isfinite(mean_nll):
        raise ValueError(f"Non-finite evaluation loss: {mean_nll}")
    return math.exp(mean_nll)

# STEP 6. Exponentiate the mean token NLL, not each chunk's loss.
base_validation_ppl = calc_perplexity(model, tokenized_validation_dataset)
base_ppl = calc_perplexity(model, tokenized_test_dataset)
print(f"Base validation perplexity: {base_validation_ppl:.4f}; test perplexity: {base_ppl:.4f}")

Base validation perplexity: 7.8430; test perplexity: 8.4954


## Training

### RTX 2070 Super / 8 GiB: time-budgeted candidate

Keep Mistral-7B in double-quantized NF4 with FP16 compute: this GPU has no native BF16 support. Use SDPA without requiring FlashAttention/Triton. Rank 16 on Q/K/V/O has the same adapter parameter count as rank 32 on Q/V for this Mistral architecture, but broader coverage; equal parameter count does not guarantee equal speed or better quality.

Try learning rate `1e-4`, cosine decay, 5% warmup, batch size 4 and up to two epochs. The old single epoch took 261.8 s; added validation and adapter changes must be measured. Context length 384 retains all current chunks (observed maximum 371 tokens before EOS), with dynamic batch padding. Do not increase context, rank, batch size, or epochs together on an 8 GiB card.

Step 8 includes validation, checkpointing, best-checkpoint reload and final adapter export. A callback requests a stop after 510 s of updates/overhead, reserving about 90 s for finalization. This is a **soft budget, not a hard deadline**: callbacks cannot interrupt a stalled CUDA step, evaluation, or filesystem write. The measured wall time below explicitly reports whether 600 s was met.

Select checkpoints by token-weighted validation NLL on a fixed 384/48/49 train/validation/test split. Test scores are for the final comparison, not selecting hyperparameters. Scores differ from the old arithmetic mean of chunk perplexities and are not directly comparable. Checkpoints contain adapters, not optimizer state; reruns start from a fresh base model, not the already-trained adapter. The exported `best_adapter` is the lowest-validation-NLL trained checkpoint in this run, not a proven global optimum.

In [12]:
from datetime import datetime
from pathlib import Path
import warnings
import numpy as np
import torch.nn.functional as F
from transformers import Trainer, TrainingArguments, TrainerCallback, EvalPrediction

def token_nll_statistics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    shifted_labels = labels[:, 1:].contiguous()
    losses = F.cross_entropy(
        logits[:, :-1, :].contiguous().float().view(-1, logits.shape[-1]),
        shifted_labels.view(-1), ignore_index=-100, reduction="none",
    ).view_as(shifted_labels)
    return torch.stack((losses.sum(dim=1), shifted_labels.ne(-100).sum(dim=1)), dim=1)

def validation_metrics(prediction: EvalPrediction) -> dict[str, float]:
    nll, count = np.asarray(prediction.predictions, dtype=np.float64).sum(axis=0)
    if count <= 0 or not np.isfinite(nll):
        raise ValueError("Validation needs finite NLL and unmasked target tokens.")
    mean_nll = float(nll / count)
    return {"token_nll": mean_nll, "perplexity": math.exp(mean_nll)}

class TrainingTimeBudget(TrainerCallback):
    def on_train_begin(self, args, state, control, **kwargs) -> None:
        _ = args, state, control, kwargs
        self.started = perf_counter()

    def on_step_end(self, args, state, control, **kwargs) -> None:
        _ = args, state, kwargs
        if perf_counter() - self.started >= 510:
            control.should_training_stop = True
            control.should_evaluate = True
            control.should_save = True

    def on_evaluate(self, args, state, control, **kwargs) -> None:
        _ = args, state, kwargs
        if perf_counter() - self.started >= 510:
            control.should_training_stop = True
            control.should_save = True

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id
model.generation_config.pad_token_id = tokenizer.pad_token_id
run_dir = Path("outputs") / datetime.now().strftime("qlora-%Y%m%d-%H%M%S-%f")
trainer = Trainer(
    model=model,
    train_dataset=tokenized_train_dataset,
    eval_dataset=tokenized_validation_dataset,
    args=TrainingArguments(
        # STEP 7. Configure for the installed Transformers 5.20 API.
        output_dir=str(run_dir),
        per_device_train_batch_size=4,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=2,
        learning_rate=1e-4,
        lr_scheduler_type="cosine",
        warmup_steps=0.05,
        weight_decay=0.01,
        max_grad_norm=1.0,
        optim="paged_adamw_8bit",
        fp16=True, bf16=False, tf32=False,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch", save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_token_nll", greater_is_better=False,
        save_total_limit=2, save_only_model=True,
        logging_steps=10, report_to="none",
        label_names=["labels"],
        seed=42, data_seed=42,
        dataloader_num_workers=0,
    ),
    data_collator=causal_lm_collator,
    processing_class=tokenizer,
    preprocess_logits_for_metrics=token_nll_statistics,
    compute_metrics=validation_metrics,
    callbacks=[TrainingTimeBudget()],
)
# STEP 8. Time training AND validation, checkpointing, reload, and export.
step8_started = perf_counter()
train_result = trainer.train()
trainer.save_model(str(run_dir / "best_adapter"))
step8_seconds = perf_counter() - step8_started
print(f"Step 8 inclusive wall time: {step8_seconds:.1f}s / 600s")
print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")
if step8_seconds > 600:
    warnings.warn("Step 8 exceeded 10 minutes. Reduce the update budget or epochs before rerunning.", RuntimeWarning)
if trainer.state.best_metric is None:
    raise RuntimeError("Training finished without a validation-selected checkpoint.")
best_validation_ppl = math.exp(trainer.state.best_metric)
print(f"Validation perplexity: base={base_validation_ppl:.4f}, best trained={best_validation_ppl:.4f}")
if best_validation_ppl >= base_validation_ppl:
    warnings.warn("Training did not improve validation perplexity over the base model.", RuntimeWarning)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Epoch,Training Loss,Validation Loss,Token Nll,Perplexity
1,1.594413,1.699839,1.702605,5.488224


Step 8 inclusive wall time: 539.7s / 600s
Best checkpoint: outputs\qlora-20261020-225342-704358\checkpoint-191
Validation perplexity: base=7.8430, best trained=5.4882


## Evaluating the finetuned model

### Measured run: 2026-10-20, RTX 2070 Super

- Inclusive Step 8: **539.7 s (9:00)**, within 600 s. The soft budget stopped at step **191/192**, or **1.9896 epochs**.
- Validation perplexity: **7.8430 base -> 5.5102 after epoch 1 -> 5.4882 best**.
- Held-out test perplexity: **8.4954 base -> 6.0018 finetuned**, a **29.35% reduction** under the corrected token-weighted metric.
- Selected checkpoint: `outputs/qlora-20261020-225342-704358/checkpoint-191`; reusable export: `outputs/qlora-20261020-225342-704358/best_adapter`.
- The seeded finetuned completion was coherent and period-styled instead of the original repeated-number failure. One prompt does not establish general generation quality.
- This run demonstrates improvement within the budget, not globally minimal perplexity. Epoch 2 improved validation perplexity only about 0.4%; a future validation-only learning-rate comparison (for example `5e-5`, `1e-4`, `2e-4`, each from fresh base weights) is more informative than blindly adding epochs. Do not select that comparison using test results.
- Checkpoint/cache/gradient warnings did not recur. The run retained the optional Triton notice and a PAD-config alignment notice; PAD IDs are now explicitly synchronized before Trainer construction. Tokenizer loading now reuses the local snapshot to avoid the observed extra Hub connection warning. These post-run setup changes do not change the trained weights.

In [13]:
# STEP 9. Generate a completion with the finetuned model and compare it to the base generation.
ft_generation = generate_text("I'm afraid I've created a ")

print("Base model generation: " + base_generation + "\n\n")
print("Finetuned generation: " + ft_generation)

Base model generation: I'm afraid I've created a 4-year old monster.

For the past 2 years, my son has played with his brother's toys - mostly Duplo and Lego. He loves to build and create things. He's never really had his own set of Lego or Duplo but recently he started asking for Lego. We gave him some small sets at Christmas (some that were on sale) but he loved them so much he wanted more. So, we got him a big box of Leg


Finetuned generation: I'm afraid I've created a 
situation of great delicacy for you. When I think of what I have done, my
heart sinks within me; but when I reflect that by some blunder, perhaps the
effect of some momentary mental aberration, I may have caused irreparable
evil to one so amiable and so beloved, my sensations are too exquisite to be
endured.”

“But how has this happened?” cried Clerval, with


Compare coherence, repetition and style across several identical prompts, not just one seeded completion. Sampling and repetition controls change decoding, **not perplexity**. Lower held-out NLL and better subjective completions are related but distinct goals. For future experiments, change one factor at a time, restart from the same base weights, keep preprocessing/splits fixed, and rank runs by validation perplexity subject to the measured 600 s limit. Reserve the test set for the chosen configuration's final report.

In [14]:
# STEP 10. Calculate the finetuned model's perplexity and compare it to the base model's.
ft_ppl = calc_perplexity(model, tokenized_test_dataset)
print(f"Base model test perplexity: {base_ppl:.4f}")
print(f"Finetuned model test perplexity: {ft_ppl:.4f}")
print(f"Relative test perplexity reduction: {100 * (1 - ft_ppl / base_ppl):.2f}%")

Base model test perplexity: 8.4954
Finetuned model test perplexity: 6.0018
Relative test perplexity reduction: 29.35%
